In [1]:
import torch
from dinosaw.helpers import ModelTypes, model_names, get_models,get_features, add_custom_font
from dinosaw.utils import do_2D_pca

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:1'
half = False

/home/ab_aimd_anja_20884/anaconda3/envs/test-multi-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import glob
glob.glob("/home/ab_aimd_anja_20884/Pawlowsky_Moritz/England/DINOMO/testing_dinov2/Dataset/IN_reduced_224_normal_Dv2")

['/home/ab_aimd_anja_20884/Pawlowsky_Moritz/England/DINOMO/testing_dinov2/Dataset/IN_reduced_224_normal_Dv2']

In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dv3', 'alibi_dv2_coco', 'nope')
models = get_models(enabled_models, "../../trained_models", DEVICE, half, conf_path='../../dinov3')
FS = 35

In [3]:
sizes = (518, 784, 1036, 1250)#, 2072)
# sizes = (1, 1.5, 2, 2.5, 3)#, 2072)
# img_fname = "wmg_si_c.png"
img_fname = "cat.jpg"


In [4]:
features = {model_key: {} for model_key in enabled_models}
features_reduced = {model_key: {} for model_key in enabled_models}
for model_key in enabled_models:
    for size in sizes:
        _img = Image.open(f"../images/{img_fname}").convert("RGB")
        # _img = _img.resize((int(size * _img.width), int(size * _img.height)), Image.LANCZOS)
        _img = _img.resize((size, size), Image.BILINEAR)
        model = models[model_key]
        feats = get_features(model, _img, False, False, device=DEVICE, to_half=half)
        features[model_key][size] = feats

In [5]:
i = 0
for model_key in enabled_models:
    for size in sizes:
        feat = features[model_key][size].copy()
        features_reduced[model_key][size] = do_2D_pca(feat.copy(), 9, pre_norm="std", post_norm='minmax')[:, :, i*3:i*3+3]

In [6]:
def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [7]:
%%capture
H,W = 24,20
NCOLS, NROWS = len(sizes), 1+ len(enabled_models)
FLIP = False
fig, axs = plt.subplots(nrows=NROWS, ncols=NCOLS, figsize=(W, H))
axs = axs.ravel()
add_custom_font("resources/fonts")
axs[0].imshow(_img)
for _ in range(NCOLS):
    axs[_].set_axis_off()
for i, model_key in enumerate(enabled_models):
    for j, size in enumerate(sizes):
        feats_red = features_reduced[model_key][size]
        # print(i, j)
        ax = axs[NCOLS*i+NCOLS+j]
        hide_axes(ax)
        ax.imshow(np.flip(feats_red, axis=2)) if FLIP else ax.imshow(feats_red)
        if j == 0:
            ax.set_ylabel(model_names[model_key], fontsize=FS, fontweight = "bold" if "alibi" in model_key else None)
        if i == 0:
            ax.set_title(f"({feats_red.shape[0]*14}, {feats_red.shape[1]*14})", fontsize = FS)
plt.tight_layout()
plt.savefig("saved/S02_size_cat.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})

## pug

In [8]:
img_fname = "pug.png"

In [9]:
features = {model_key: {} for model_key in enabled_models}
features_reduced = {model_key: {} for model_key in enabled_models}
for model_key in enabled_models:
    for size in sizes:
        _img = Image.open(f"../images/{img_fname}").convert("RGB")
        # _img = _img.resize((int(size * _img.width), int(size * _img.height)), Image.LANCZOS)
        _img = _img.resize((size, size), Image.BILINEAR)
        model = models[model_key]
        feats = get_features(model, _img, False, False, device=DEVICE, to_half=half)
        features[model_key][size] = feats

In [10]:
i = 0
for model_key in enabled_models:
    for size in sizes:
        feat = features[model_key][size].copy()
        features_reduced[model_key][size] = do_2D_pca(feat.copy(), 9, pre_norm="std", post_norm='minmax')[:, :, i*3:i*3+3]

In [11]:
%%capture
H,W = 24,20
NCOLS, NROWS = len(sizes), 1+ len(enabled_models)

FLIP = False
fig, axs = plt.subplots(nrows=NROWS, ncols=NCOLS, figsize=(W, H))
axs = axs.ravel()
add_custom_font("resources/fonts")
axs[0].imshow(_img)
for _ in range(NCOLS):
    axs[_].set_axis_off()
for i, model_key in enumerate(enabled_models):
    for j, size in enumerate(sizes):
        feats_red = features_reduced[model_key][size]
        # print(i, j)
        ax = axs[NCOLS*i+NCOLS+j]
        hide_axes(ax)
        ax.imshow(np.flip(feats_red, axis=2)) if FLIP else ax.imshow(feats_red)
        if j == 0:
            ax.set_ylabel(model_names[model_key], fontsize=FS, fontweight = "bold" if "alibi" in model_key else None)
        if i == 0:
            ax.set_title(f"({feats_red.shape[0]*14}, {feats_red.shape[1]*14})", fontsize = FS)
plt.tight_layout()
plt.savefig("saved/S02_size_pug.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})

## combine

In [12]:
%%capture
W, H = 4, 2.5
FS = 30
add_custom_font('resources/fonts', 'Grotesk')

N_ROWS, N_COLS = 4, 7

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(W * N_COLS, H * N_COLS))

fig_a = Image.open('saved/S02_size_cat.jpeg')
fig_b = Image.open('saved/S02_size_pug.jpeg')

axs[0].imshow(fig_a)
axs[0].axis('off')
axs[1].imshow(fig_b)
axs[1].axis('off')


labels = ['(a)', '(b)']
y_off = [1.01, 1.02]
# print(gs.subplots()[0, 0])
for i in range(2):
    ax = axs[i]
    ax.text(-0.075, y_off[i], labels[i], transform=ax.transAxes,
            fontsize=FS+6, fontweight='bold', color='black')

plt.tight_layout()
plt.savefig("saved/S02_combined.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})